# 07 — ZenML Pipelines

- Trigger training and inference pipelines from notebook
- Inspect ZenML artifact store
- Show pipeline DAG and run comparison

**Prerequisites:** `zenml init` run in project root. `zenml up` for dashboard.

```bash
zenml init      # run once in energy-price-forecast/
zenml up        # http://127.0.0.1:8237
```

In [ ]:
import sys
sys.path.insert(0, '..')

from zenml.client import Client

client = Client()
print('ZenML version:', client.zen_store.info.version)
print('Active stack:', client.active_stack_model.name)

## 7.1 Run training pipeline

In [ ]:
from pipelines.training_pipeline import training_pipeline

# ZenML will track all steps, artifacts, and metrics
training_pipeline()
print('Training pipeline complete. View run at http://127.0.0.1:8237')

## 7.2 Inspect pipeline runs

In [ ]:
runs = client.list_pipeline_runs(pipeline_name_or_id='training_pipeline', size=5)
print(f'Found {len(runs)} run(s):')
for run in runs:
    print(f'  [{run.status}] {run.name} — {run.start_time}')

## 7.3 Inspect artifacts from latest run

In [ ]:
if runs:
    latest_run = runs[0]
    print(f'Latest run: {latest_run.name}')
    for step_name, step in latest_run.steps.items():
        print(f'\n  Step: {step_name} [{step.status}]')
        for output_name, output in step.outputs.items():
            print(f'    Output "{output_name}": {output.type} (version {output.version})')

## 7.4 Compare two runs (metrics)

In [ ]:
import json
from pathlib import Path

metrics_path = Path('../models/metrics.json')
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print('Latest run metrics:')
    for k, v in list(metrics.items())[:8]:
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## 7.5 Run inference pipeline

In [ ]:
from pipelines.inference_pipeline import inference_pipeline

inference_pipeline()
print('Inference pipeline complete.')

# Show latest prediction file
import duckdb
import glob
pred_files = sorted(glob.glob('../data/predictions/*.parquet'))
if pred_files:
    df = duckdb.connect().execute(f"SELECT * FROM read_parquet('{pred_files[-1]}')").fetchdf()
    print(f'Latest forecast: {pred_files[-1]}')
    print(df[['prediction_ts_utc'] + [c for c in df.columns if 'price_t_plus' in c][:5]].head())